In [1]:
clean_file = '2_clean_FAME_data/firms_final_sample_fixed_vars.csv'

In [2]:
# Get column names
import pandas as pd

# Import column names from csv file and save into array
col_names = pd.read_csv(clean_file, nrows=1)
col_array = col_names.columns.tolist()
# Print each on separate line
for i, col in enumerate(col_array):
    print(f"{i}: {col}")

0: registered_number
1: company_name_A
2: primary_uk_sic_2007_code
3: primary_uk_sic_2007_description
4: full_overview
5: primary_business_line
6: no_of_available_years
7: latest_accounts_date
8: company_name_B
9: inactive
10: quoted
11: own_data
12: woco
13: bv_d_id_number
14: company_status
15: status_date
16: legal_form
17: date_of_incorporation
18: accounting_reference_date
19: registered_accounts_type
20: jordans_company_classification
21: account_currency
22: guo_name
23: guo_bv_d_id_number
24: duo_name
25: duo_bv_d_id_number


In [ ]:
# Import 2_clean_FAME_data/firms_final_sample_fixed_vars.csv
# As this is just a large file, only import columns 1-5

df = pd.read_csv(clean_file, usecols=range(5))

# Summarise the data
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 452638 entries, 0 to 452637
Data columns (total 5 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   registered_number                452638 non-null  object 
 1   company_name_A                   452638 non-null  str    
 2   primary_uk_sic_2007_code         447227 non-null  float64
 3   primary_uk_sic_2007_description  447216 non-null  str    
 4   full_overview                    121455 non-null  str    
dtypes: float64(1), object(1), str(3)
memory usage: 17.3+ MB
None


C:\Users\lazym\AppData\Local\Temp\ipykernel_13152\2496378275.py:5: DtypeWarning: Columns (0: registered_number) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('2_clean_FAME_data/firms_final_sample_fixed_vars.csv', usecols=range(5))


In [7]:
mycols = [4, 5, 6, 8, 9, 12, 13, 14, 21, 22, 23, 24, 25]
# For every index value in mycols, keep the corresponding column name from col_names
col_subset = [col_array[i] for i in mycols]

print(col_subset)

# from the same CSV, import columns 5, 6, 7, 9, 10, 13, 14, 15, 22, 23, 24, 25, 26, and 27. Just the first 5000 rows is fine.
# Correct for usecols do not match columns, columns expected but not found: [4, 5, 6...]

df_subset = pd.read_csv(clean_file, usecols=[i - 1 for i in mycols])
print(df_subset.head())

['full_overview', 'primary_business_line', 'no_of_available_years', 'company_name_B', 'inactive', 'woco', 'bv_d_id_number', 'company_status', 'account_currency', 'guo_name', 'guo_bv_d_id_number', 'duo_name', 'duo_bv_d_id_number']
                     primary_uk_sic_2007_description full_overview  \
0         Security and commodity contracts brokerage           NaN   
1  Manufacture of communication equipment (other ...           NaN   
2                  Other mining and quarrying n.e.c.           NaN   
3                                 Non-life insurance           NaN   
4                                 Non-life insurance           NaN   

  primary_business_line latest_accounts_date           company_name_B  \
0                   NaN           2016-06-30       BLACKSTAR GROUP SE   
1                   NaN           2011-12-31          SCOTTY GROUP SE   
2                   NaN           2017-06-30       PETRA DIAMONDS LTD   
3                   NaN           2023-12-31             

In [11]:
# Frequency table of the values in the 'woco' column (counts)
woco_freq = df_subset['woco'].value_counts()
# Add an additional column for percentages
woco_freq = woco_freq.to_frame().reset_index()
woco_freq.columns = ['woco', 'count']
woco_freq['percentage'] = (woco_freq['count'] / woco_freq['count'].sum()) * 100
print(woco_freq)    

  woco   count  percentage
0   No  447740   99.358235
1  Yes    2892    0.641765


Upsert extract to avoid adding duplicate registered_numbers.
Instead we fast add rows and then duplication check at the end

In [ ]:
# 2. UPSERT LOGIC: keep existing records and add new records as new rows, and then after update with the union
table_fame_fixed: ibis.expr.types.Table = con.table("fame_fixed")
table_vjoined_fixed_raw = table_fame_fixed.anti_join(
    table_t_fixed_cast,
    "registered_number"
)
table_updated_fixed_raw = table_vjoined_fixed_raw.union(table_t_fixed_cast)
con.create_table("fame_fixed", table_updated_fixed_raw, overwrite=True)
print(f"✅ Successfully processed fixed table from: {ind}")